In [ ]:
import json
import os
import sqlite3
import pandas as pd

# ============================================================
# 1. MEMBACA DATA EXCEL SUMBER
# ============================================================
# Using the correct path for the Excel file from Google Drive
file_excel = "/content/drive/MyDrive/data wrangling/data_tugas_data_wrangling.xlsx"

df_pos = pd.read_excel(file_excel, sheet_name="Transaksi_POS")
df_crm = pd.read_excel(file_excel, sheet_name="CRM_Pelanggan")
df_logistik = pd.read_excel(file_excel, sheet_name="Logistik_API")

# Hapus spasi tak terlihat di nama kolom
df_pos.columns = df_pos.columns.str.strip()
df_crm.columns = df_crm.columns.str.strip()
df_logistik.columns = df_logistik.columns.str.strip()

print("Data berhasil dibaca dari Excel.")
print("Jumlah transaksi :", len(df_pos))
print("Jumlah pelanggan :", len(df_crm))
print("Jumlah logistik :", len(df_logistik))

# ============================================================
# 2. MEMBUAT FILE CSV DARI DATA POS
# ============================================================
df_pos.to_csv("transaksi_penjualan.csv", index=False)
print("\nFile transaksi_penjualan.csv berhasil dibuat.")

# ============================================================
# 3. LOAD DATA TRANSAKSI DARI CSV
# ============================================================
df_transaksi = pd.read_csv(
    "transaksi_penjualan.csv", dtype={"order_id": str, "customer_id": str}
)

df_transaksi["order_date"] = pd.to_datetime(
    df_transaksi["order_date"],
    errors="coerce"
)

# Bersihkan spasi
df_transaksi["order_id"] = df_transaksi["order_id"].astype(str).str.strip()
df_transaksi["customer_id"] = df_transaksi["customer_id"].astype(str).str.strip()

print("\n=== DATA TRANSAKSI ===")
print(df_transaksi.head())

# ============================================================
# 4. MEMBUAT DATABASE SQLITE DARI DATA CRM
# ============================================================
conn = sqlite3.connect("crm_membership.db")
df_crm.to_sql("customers", conn, if_exists="replace", index=False)
conn.close()
print("\nDatabase crm_membership.db berhasil dibuat.")

# ============================================================
# 5. MENGAMBIL DATA CUSTOMER YANG AKTIF
# ============================================================
conn = sqlite3.connect("crm_membership.db")
query = """
SELECT
    customer_id,
    customer_name,
    city,
    membership_level,
    is_active
FROM customers
WHERE is_active = 1
"""
df_pelanggan = pd.read_sql_query(query, conn)
conn.close()

df_pelanggan["customer_id"] = df_pelanggan["customer_id"].astype(str).str.strip()

print("\n=== DATA CUSTOMER AKTIF ===")
print(df_pelanggan.head())

# ============================================================
# 6. MEMBUAT DATA LOGISTIK DALAM FORMAT JSON
# ============================================================
data_logistik_json = {"shipments": []}

for _, row in df_logistik.iterrows():
    order_id_clean = str(row["order_id"]).split(".")[0].strip()
    data_logistik_json["shipments"].append({
        "order_id": order_id_clean,
        "shipping": {
            "courier": str(row["courier"]),
            "status": str(row["shipment_status"]),
            "updated_at": str(row["shipment_updated_at"]),
        },
    })

with open("mock_logistics.json", "w", encoding="utf-8") as file:
    json.dump(data_logistik_json, file, indent=4, ensure_ascii=False)

print("\nFile mock_logistics.json berhasil dibuat.")

# ============================================================
# 7. MEMBACA DAN FLATTEN JSON LOGISTIK
# ============================================================
with open("mock_logistics.json", "r", encoding="utf-8") as file:
    data_logistik = json.load(file)

df_logistik_flat = pd.json_normalize(data_logistik["shipments"])
df_logistik_flat = df_logistik_flat.rename(
    columns={
        "shipping.courier": "courier",
        "shipping.status": "shipment_status",
        "shipping.updated_at": "shipment_updated_at",
    }
)
df_logistik_flat["order_id"] = df_logistik_flat["order_id"].astype(str).str.strip()

# ============================================================
# 8. MERGE DATA MASTER
# ============================================================
df_master_analisis = df_transaksi.merge(
    df_pelanggan, on="customer_id", how="inner"
)
df_master_analisis = df_master_analisis.merge(
    df_logistik_flat, on="order_id", how="left"
)

print("\n================================================")
print("DATA MASTER ANALISIS HASIL MERGE")
print("================================================")
print(df_master_analisis.head())
print("\nJumlah baris :", len(df_master_analisis))
print("Jumlah kolom :", len(df_master_analisis.columns))

# ============================================================
# 9. CEK TRANSAKSI KUPON & PENGIRIMAN
# ============================================================
df_transaksi_kupon = df_master_analisis[
    df_master_analisis["coupon_code"].notna()
]

print("\nTotal transaksi menggunakan kupon:", len(df_transaksi_kupon))
print("\nPROSES DATA WRANGLING SELESAI.")

Data berhasil dibaca dari Excel.
Jumlah transaksi : 20
Jumlah pelanggan : 22
Jumlah logistik : 20

File transaksi_penjualan.csv berhasil dibuat.

=== DATA TRANSAKSI ===
  order_id customer_id order_date  total_amount coupon_code
0   ORD001        C001 2026-09-20        125000     HEMAT10
1   ORD002        C002 2026-09-20        275000     HEMAT10
2   ORD003        C003 2026-09-21        185000         NaN
3   ORD004        C004 2026-09-21        320000     HEMAT10
4   ORD005        C005 2026-09-22         95000         NaN

Database crm_membership.db berhasil dibuat.

=== DATA CUSTOMER AKTIF ===
  customer_id  customer_name        city membership_level  is_active
0        C001     Alya Putri    Pemalang             Gold          1
1        C002     Nisa Rahma       Tegal           Silver          1
2        C003   Rani Lestari  Pekalongan           Bronze          1
3        C004  Dina Maharani      Brebes             Gold          1
4        C005  Salsa Nuraini    Pemalang           S

In [ ]:
# ============================================================
# BAGIAN C - KAMUS DATA
# ============================================================

def buat_kamus_data(df):
    kamus = []

    for kolom in df.columns:
        kamus.append({
            "nama_kolom": kolom,
            "tipe_data": str(df[kolom].dtype),
            "jumlah_missing": df[kolom].isna().sum(),
            "persentase_missing": round(
                df[kolom].isna().mean() * 100, 2
            ),
            "jumlah_unique": df[kolom].nunique(),
            "contoh_nilai": ", ".join(
                df[kolom]
                .dropna()
                .astype(str)
                .unique()[:3]
            )
        })

    return pd.DataFrame(kamus)


# Membuat kamus data
df_kamus = buat_kamus_data(df_master_analisis)

# Menyimpan ke CSV
df_kamus.to_csv(
    "kamus_data_hasil.csv",
    index=False
)

print("================================================")
print("KAMUS DATA")
print("================================================")
print(df_kamus)

print("\nFile kamus_data_hasil.csv berhasil dibuat.")

KAMUS DATA
             nama_kolom       tipe_data  jumlah_missing  persentase_missing  \
0              order_id          object               0                0.00   
1           customer_id          object               0                0.00   
2            order_date  datetime64[ns]               0                0.00   
3          total_amount           int64               0                0.00   
4           coupon_code          object               6               31.58   
5         customer_name          object               0                0.00   
6                  city          object               0                0.00   
7      membership_level          object               0                0.00   
8             is_active           int64               0                0.00   
9               courier          object               0                0.00   
10      shipment_status          object               0                0.00   
11  shipment_updated_at          object  

In [ ]:
# ============================================================
# VALIDASI DATA DENGAN ASSERTION
# ============================================================

# 1. Tidak boleh ada order_id yang duplikat
assert df_master_analisis["order_id"].is_unique, \
    "Terdapat order_id yang duplikat!"

print("✓ Tidak ada order_id duplikat.")


# 2. total_amount tidak boleh negatif
assert (df_master_analisis["total_amount"] >= 0).all(), \
    "Terdapat total_amount yang negatif!"

print("✓ Tidak ada total_amount negatif.")


# 3. order_date tidak boleh berada di masa depan
assert (
    df_master_analisis["order_date"] <= pd.Timestamp.today()
).all(), \
    "Terdapat order_date yang berada di masa depan!"

print("✓ Tidak ada order_date di masa depan.")


print("\n================================================")
print("SEMUA VALIDASI BERHASIL")
print("================================================")

✓ Tidak ada order_id duplikat.
✓ Tidak ada total_amount negatif.
✓ Tidak ada order_date di masa depan.

SEMUA VALIDASI BERHASIL
